# Module 07 -- Gamma Exposure (GEX)

**Author: Djellal Djouad** -- CrossVol Research | [crossvol.com](https://crossvol.com) | ORCID [0009-0002-4911-1118](https://orcid.org/0009-0002-4911-1118)

GEX is probably the most popular retail quant metric right now. Every
fintwit account posts a GEX chart. Most of them get it half right and
draw half-wrong conclusions. This module shows you the real formula,
what it tells you, and -- more importantly -- what it does not.

I spent years watching the GEX number move on my screen while the
market did something completely different. The metric has value, but
only if you understand its blind spots.

---
*License: MIT with Educational Use Clause -- see LICENSE. Not trading advice.*


In [ ]:
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt


## The GEX Formula

For each strike $K_i$, gamma exposure is:

$$\text{GEX}_i = \Gamma_i \times \text{OI}_i \times 100 \times S$$

where $\Gamma_i$ is Black-Scholes gamma at that strike, $\text{OI}_i$ is
open interest (calls positive, puts negative from the dealer's perspective),
$100$ is the contract multiplier, and $S$ is spot.

The sign convention matters. Dealers are generally **short** the options
that retail and funds buy. When a customer buys a call, the dealer is
short that call, so dealer gamma is positive for calls (they bought
the underlying to hedge) and negative for puts.


In [ ]:
# Black-Scholes gamma -- same formula from Module 01
def bs_gamma(S, K, T, r, sigma, q=0):
    """Black-Scholes gamma for a European option."""
    d1 = (np.log(S / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    return np.exp(-q * T) * norm.pdf(d1) / (S * sigma * np.sqrt(T))


In [ ]:
# Generate synthetic OI data.
# In real life you pull this from CBOE or your data vendor.
# The distribution is roughly bell-shaped around ATM with a put skew --
# more put OI below the money, more call OI above.

np.random.seed(42)
spot = 5500
strikes = np.arange(5000, 6001, 25)

# Synthetic call OI -- peaks near ATM, decays outward
call_oi = np.exp(-0.5 * ((strikes - spot) / 200)**2) * 15000
call_oi = (call_oi + np.random.uniform(0, 2000, len(strikes))).astype(int)

# Synthetic put OI -- shifted below spot (hedging demand)
put_oi = np.exp(-0.5 * ((strikes - (spot - 100)) / 250)**2) * 18000
put_oi = (put_oi + np.random.uniform(0, 2500, len(strikes))).astype(int)

print(f"Total call OI: {call_oi.sum():,}")
print(f"Total put OI:  {put_oi.sum():,}")
print(f"Put/Call OI ratio: {put_oi.sum() / call_oi.sum():.2f}")


## Computing the GEX Profile

We compute gamma at each strike, then multiply by OI and spot.
The key assumption: dealers are net short everything. This is a
simplification -- some strikes have dealer-long positioning -- but
it is the standard public approach.


In [ ]:
T = 30 / 365   # assume 30 DTE for the whole chain (another simplification)
r = 0.05
sigma = 0.18

gammas = np.array([bs_gamma(spot, K, T, r, sigma) for K in strikes])

# Dealer GEX: calls contribute positive gamma, puts contribute negative
# (dealer is short the customer's position, so short calls = long gamma,
#  short puts = short gamma)
gex_calls = gammas * call_oi * 100 * spot
gex_puts = -gammas * put_oi * 100 * spot   # negative for puts
gex_total = gex_calls + gex_puts

# Scale to billions for readability
gex_calls_bn = gex_calls / 1e9
gex_puts_bn = gex_puts / 1e9
gex_total_bn = gex_total / 1e9

print(f"Net GEX: ${gex_total.sum() / 1e9:.2f}B")


## GEX Profile Chart

This is the chart you see everywhere. Positive bars mean dealer
long gamma at that strike -- they will sell into rallies and buy
dips, dampening moves. Negative bars mean dealer short gamma --
they will chase the move, amplifying it.


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

ax.bar(strikes, gex_calls_bn, width=20, alpha=0.6, color='#2196F3', label='Call GEX')
ax.bar(strikes, gex_puts_bn, width=20, alpha=0.6, color='#F44336', label='Put GEX')
ax.plot(strikes, gex_total_bn, color='white', linewidth=2, marker='o',
        markersize=3, label='Net GEX', zorder=5)

ax.axhline(0, color='gray', linewidth=0.8)
ax.axvline(spot, color='#FFD700', linewidth=2, linestyle='--', label=f'Spot ({spot})')

ax.set_xlabel('Strike', fontsize=12)
ax.set_ylabel('Gamma Exposure ($B)', fontsize=12)
ax.set_title('Dealer Gamma Exposure by Strike', fontsize=14)
ax.legend(fontsize=10)
ax.set_facecolor('#1a1a2e')
fig.patch.set_facecolor('#16213e')
ax.tick_params(colors='white')
ax.xaxis.label.set_color('white')
ax.yaxis.label.set_color('white')
ax.title.set_color('white')
ax.legend(facecolor='#1a1a2e', edgecolor='gray', labelcolor='white')

plt.tight_layout()
plt.savefig('../data/07_gex_profile.png', dpi=100, bbox_inches='tight')
plt.show()


## The GEX Flip Point

The flip point is where net GEX crosses zero. Above it, dealers are
long gamma and the market is "pinned" -- moves get dampened. Below it,
dealers are short gamma and the market accelerates -- moves get amplified.

This is the single most useful output from a GEX analysis. Not the
absolute number, but where the regime change happens.


In [ ]:
# Find the flip point -- where net GEX crosses zero
flip_idx = None
flip_strike = None
for i in range(len(gex_total) - 1):
    if gex_total[i] * gex_total[i + 1] < 0:
        # Linear interpolation
        w = abs(gex_total[i]) / (abs(gex_total[i]) + abs(gex_total[i + 1]))
        flip_strike = strikes[i] + w * (strikes[i + 1] - strikes[i])
        flip_idx = i
        break

if flip_strike is not None:
    print(f"GEX flip point: {flip_strike:.0f}")
    print(f"Spot is {'above' if spot > flip_strike else 'below'} the flip")
    print(f"Regime: {'long gamma (dampening)' if spot > flip_strike else 'short gamma (amplifying)'}")
else:
    print("No flip point found in this range")


## Where GEX Gets It Wrong

Here is where most people stop. They see the GEX chart and think
they understand dealer positioning. They do not.

**The 30-50% error margin is real.** I have seen it on the desk for
years. The GEX number says the market should pin and it gaps 2%.
Why? Because vanilla gamma is only one of the forces:

1. **Vanna** -- delta sensitivity to vol. When vol drops, dealers
   who are short calls see their delta shrink, so they sell futures.
   This can overpower gamma entirely during a vol crush.

2. **Charm** -- delta decay. As options approach expiry, the delta
   changes just from time passing. This creates flows that are
   invisible to a pure gamma analysis.

3. **Volga/vomma** -- gamma of vega. When vol moves, the hedging
   of vega exposure creates its own flows.

4. **OI assumptions** -- we assumed all OI is customer long / dealer
   short. In reality, a chunk of OI is interdealer or spread trades
   where the net gamma is much smaller.

GEX is the first lens. You need at least four to see the full picture.

For the full four-lens framework (gamma + vanna + charm + volga),
see [Beyond Gamma Exposure](https://www.amazon.com/dp/B0H2RZGMY6)
and the [working paper](https://doi.org/10.5281/zenodo.20509786).

**We are not implementing vanna or volga here.** That is intentional.
The formulas require dealer inventory assumptions that go beyond
what public data can tell you. But knowing that the gap exists
is already more than most people trading off a GEX chart understand.


## Gamma Exposure Over Time

One more thing that is often missed: GEX changes every day even
if OI stays constant, because gamma itself is a function of
time to expiry and spot. Let us visualize how the profile shifts.


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

for dte, alpha in [(45, 0.3), (30, 0.5), (14, 0.7), (7, 0.9), (1, 1.0)]:
    t = dte / 365
    g = np.array([bs_gamma(spot, K, t, r, sigma) for K in strikes])
    net_gex = (g * call_oi * 100 * spot - g * put_oi * 100 * spot) / 1e9
    ax.plot(strikes, net_gex, linewidth=1.5, alpha=alpha, label=f'{dte} DTE')

ax.axhline(0, color='gray', linewidth=0.8)
ax.axvline(spot, color='#FFD700', linewidth=1.5, linestyle='--')
ax.set_xlabel('Strike', fontsize=12)
ax.set_ylabel('Net GEX ($B)', fontsize=12)
ax.set_title('GEX Profile by Days to Expiry -- Same OI', fontsize=14)
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('../data/07_gex_time.png', dpi=100, bbox_inches='tight')
plt.show()


Notice how gamma concentrates near ATM as expiry approaches.
On expiration day, the gamma spike at the pinning strike is enormous.
This is why OpEx days are volatile -- the hedging flows become
massive and concentrated in a narrow range.

---

**Next:** [Module 08 -- Exotic Payoffs](08_exotics.py)

---
*Djellal Djouad -- CrossVol Research -- 2026*
